# 02 — Generator Selection

Detailed operational description: [`docs/PROTOCOL.md`, sections 3–5](../../docs/PROTOCOL.md#3-post-benchmark-gate-calibration-audit).

Select one fine-tuned and one from-scratch generator under the approved post-benchmark amendment (Option B). This notebook reads results only; it loads no feature encoder, regenerates no images, and never accesses the test split.

## Load benchmark results, registry and active amendment

In [1]:
from pathlib import Path
import json
import sys
import csv
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import load_protocol, load_registry, rank_generator_family
from notebooks.utility import gate_audit as ga
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
amendment = ga.load_active_amendment(ROOT)
confirmed_by_gen = ga.confirmed_duplicate_rates(ROOT)
canonical_metrics_path = ROOT / protocol['outputs']['metrics']
corrected_metrics_path = canonical_metrics_path.with_name('generator_summary_corrected.csv')
metrics_path = corrected_metrics_path if corrected_metrics_path.is_file() else canonical_metrics_path
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
paired_path = ROOT / protocol['outputs']['paired_differences']
if not paired_path.is_file():
    paired_path = ROOT / protocol['outputs']['root'] / 'gate_audit/paired_generator_differences.csv'
paired_rows = list(csv.DictReader(paired_path.open())) if paired_path.is_file() else []
{'active_amendment': protocol.get('active_amendment'),
 'amendment_status': amendment['status'] if amendment else None,
 'selected_policy': amendment['selected_policy'] if amendment else None,
 'post_benchmark': amendment['status'] == 'approved_post_benchmark' if amendment else None,
 'benchmark_summary_source': str(metrics_path.relative_to(ROOT)),
 'n_benchmark_rows': len(benchmark_rows), 'confirmed_duplicate_rates': confirmed_by_gen} if benchmark_rows else 'Not yet evaluated'


{'active_amendment': 'configs/generator_benchmark_protocol_amendment_v1.json',
 'amendment_status': 'approved_post_benchmark',
 'selected_policy': 'B',
 'post_benchmark': True,
 'benchmark_summary_source': 'results/2_diffusers/benchmark/generator_summary_corrected.csv',
 'n_benchmark_rows': 12,
 'confirmed_duplicate_rates': {'02_sd21_filtered_100steps': 0.0,
  '03_sd21_vae_finetuned': 0.0,
  '04_sd21_lora': 0.0,
  '07_ldm_sdvae_extra1361': 0.0,
  '08_ldm_v3_sdvae_fromscratch': 0.0}}

## Original gates outcome vs amended Option B safety gates

The original outcome (zero eligible under the preregistered coverage/pHash gates) is shown alongside the amended safety-gate eligibility. Coverage and pHash-only similarity are descriptive under the amendment, not binary gates.

In [2]:
filtered_rows = [row for row in benchmark_rows if row.get('condition') == 'FILTERED']
original_finetuned = rank_generator_family(filtered_rows, 'finetuned', protocol['eligibility_gates']) if filtered_rows else []
original_fromscratch = rank_generator_family(filtered_rows, 'from_scratch', protocol['eligibility_gates']) if filtered_rows else []
amended_finetuned = ga.amended_family_ranking(filtered_rows, 'finetuned', amendment, confirmed_by_gen) if (filtered_rows and amendment) else []
amended_fromscratch = ga.amended_family_ranking(filtered_rows, 'from_scratch', amendment, confirmed_by_gen) if (filtered_rows and amendment) else []
original_outcome = {'eligible_under_original_gates': sum(bool(row['eligible']) for row in original_finetuned + original_fromscratch),
                    'exclusions': [(row['generator_id'], row['exclusion_reasons']) for row in original_finetuned + original_fromscratch]}
amended_outcome = {'eligible_under_amended_safety_gates': sum(bool(row['eligible']) for row in amended_finetuned + amended_fromscratch),
                   'amended_exclusions': [(row['generator_id'], row['amended_exclusion_reasons']) for row in amended_finetuned + amended_fromscratch],
                   'finetuned_rank': [(row['generator_id'], row['family_rank']) for row in amended_finetuned],
                   'from_scratch_rank': [(row['generator_id'], row['family_rank']) for row in amended_fromscratch]}
{'original': original_outcome, 'amended': amended_outcome}


{'original': {'eligible_under_original_gates': 0,
  'exclusions': [('02_sd21_filtered_100steps',
    ['perceptual_duplicate_rate', 'rad_dino_coverage']),
   ('03_sd21_vae_finetuned',
    ['perceptual_duplicate_rate', 'rad_dino_coverage']),
   ('04_sd21_lora', ['perceptual_duplicate_rate', 'rad_dino_coverage']),
   ('07_ldm_sdvae_extra1361',
    ['perceptual_duplicate_rate', 'rad_dino_coverage']),
   ('08_ldm_v3_sdvae_fromscratch',
    ['perceptual_duplicate_rate', 'rad_dino_coverage']),
   ('05_ldm_basic_fromscratch',
    ['perceptual_duplicate_rate', 'rad_dino_coverage', 'registry_role'])]},
 'amended': {'eligible_under_amended_safety_gates': 5,
  'amended_exclusions': [('02_sd21_filtered_100steps', []),
   ('03_sd21_vae_finetuned', []),
   ('04_sd21_lora', []),
   ('07_ldm_sdvae_extra1361', []),
   ('08_ldm_v3_sdvae_fromscratch', []),
   ('05_ldm_basic_fromscratch',
    ['confirmed_duplicate_rate', 'registry_role'])],
  'finetuned_rank': [('02_sd21_filtered_100steps', 1),
   ('03_sd2

## Descriptive metrics and KID-primary ranking

In [3]:
display_columns = ['generator_id', 'family_rank', 'raddino_kid', 'raddino_kid_stability_low', 'raddino_kid_stability_high',
                   'raddino_coverage', 'raddino_precision', 'raddino_fid', 'inception_kid', 'raddino_kid_std',
                   'confirmed_duplicate_rate', 'perceptual_hash_duplicate_rate',  # descriptive only, not gates
                   'train_memorization_rate', 'synthetic_exact_duplicate_rate', 'provenance_manifest_valid',
                   'lineage_complete', 'training_corpus_manifest', 'generation_seconds_per_image', 'efficiency_status']
[[{column: row.get(column) for column in display_columns} for row in ranking] for ranking in (amended_finetuned, amended_fromscratch)]


[[{'generator_id': '02_sd21_filtered_100steps',
   'family_rank': 1,
   'raddino_kid': '0.1990891096955254',
   'raddino_kid_stability_low': '0.17645359028119864',
   'raddino_kid_stability_high': '0.22257210116478848',
   'raddino_coverage': '0.136986301369863',
   'raddino_precision': '0.3287671232876712',
   'raddino_fid': '65.48050878189207',
   'inception_kid': '0.055778061394022416',
   'raddino_kid_std': '0.01179213321210437',
   'confirmed_duplicate_rate': 0.0,
   'perceptual_hash_duplicate_rate': '0.10874357090374724',
   'train_memorization_rate': '0.0',
   'synthetic_exact_duplicate_rate': '0.0',
   'provenance_manifest_valid': 'True',
   'lineage_complete': 'True',
   'training_corpus_manifest': 'results/2_diffusers/provenance/runtime/shared/rsna_train_real_plus_positive_augmentation.csv',
   'generation_seconds_per_image': '',
   'efficiency_status': 'unavailable_invalid_duration_semantics'},
  {'generator_id': '03_sd21_vae_finetuned',
   'family_rank': 2,
   'raddino_kid'

## Paired differences and manual selection under the amendment

In [4]:
SELECTED_FINETUNED_GENERATOR = "02_sd21_filtered_100steps"
SELECTED_FROM_SCRATCH_GENERATOR = "07_ldm_sdvae_extra1361"
PROPOSED_FINETUNED_GENERATOR = next((row['generator_id'] for row in amended_finetuned if row['eligible']), None)
PROPOSED_FROM_SCRATCH_GENERATOR = next((row['generator_id'] for row in amended_fromscratch if row['eligible']), None)
SELECTION_NOTES = ('Post-benchmark amendment v1 (Option B), human-approved: coverage-point and pHash-only removed as binary gates; '
                   'safety gates retained (exact/confirmed duplicate, train memorization, corruption, FILTERED validity, provenance, '
                   'lineage, test access). All five official candidates pass the safety gates; the preregistered KID-primary hierarchy '
                   'selects G02 (fine-tuned) and G07 (from-scratch). Downstream results must be interpreted with this amendment stated.')
{'selected': (SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR),
 'amended_proposed_top_rank': (PROPOSED_FINETUNED_GENERATOR, PROPOSED_FROM_SCRATCH_GENERATOR),
 'paired_generator_differences': paired_rows}


{'selected': ('02_sd21_filtered_100steps', '07_ldm_sdvae_extra1361'),
 'amended_proposed_top_rank': ('02_sd21_filtered_100steps',
  '07_ldm_sdvae_extra1361'),
 'paired_generator_differences': [{'family': 'finetuned',
   'left_generator': 'G02',
   'right_generator': 'G03',
   'mean_paired_kid_difference': '-0.0780561380994298',
   'median_paired_kid_difference': '-0.07993368146820234',
   'stability_interval_low': '-0.13958757330199764',
   'stability_interval_high': '-0.020937478709958577',
   'left_win_fraction': '1.0',
   'right_win_fraction': '0.0',
   'practical_equivalence_margin': '0.001',
   'practically_similar': 'False'},
  {'family': 'finetuned',
   'left_generator': 'G02',
   'right_generator': 'G04',
   'mean_paired_kid_difference': '-0.08980888068034877',
   'median_paired_kid_difference': '-0.08951053919156204',
   'stability_interval_low': '-0.12550596180533735',
   'stability_interval_high': '-0.05663624356226625',
   'left_win_fraction': '1.0',
   'right_win_fraction'

## Validate and save the selection (post-benchmark amendment)

In [5]:
SAVE_SELECTION = True
if SAVE_SELECTION and benchmark_rows and amendment:
    output = ga.save_amended_selection(ROOT, SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                       benchmark_rows, notes=SELECTION_NOTES)
    print('Saved selection to', output)
    print(json.dumps(json.loads(Path(output).read_text()), indent=1))
else:
    print('Selection not saved. Requires benchmark results and an active amendment.')


Saved selection to /mnt/MammoDiffusion/MammoDiffusion/configs/selected_generators.json
{
 "finetuned": "02_sd21_filtered_100steps",
 "from_scratch": "07_ldm_sdvae_extra1361",
 "schema_version": 2,
 "primary_metric": "raddino_kid",
 "benchmark_HEAD": "cd05886c0e7044325063d9e2db4bf2de6d285dc4",
 "benchmark_run_id": "generator_benchmark_20260714T221055Z_cd05886c",
 "benchmark_summary_path": "results/2_diffusers/benchmark/generator_summary_corrected.csv",
 "benchmark_summary_sha256": "a64e98b4ad3b0a9aedfc1ec8bc3ff53e6ada6a6b410f6c23373adac1f2179200",
 "amended_gate_results_path": "results/2_diffusers/benchmark/gate_audit/amended_gate_results.csv",
 "amended_gate_results_sha256": "244bcb75774164a168b2ac512faa2c4e367e9f6d37bb35bc2757c528f4955f5e",
 "active_amendment": "configs/generator_benchmark_protocol_amendment_v1.json",
 "active_amendment_sha256": "d7179084f10a8aa9e7c5160215a11eccbb991a7b9b0b7994366c564586de215a",
 "selection_evidence_path": "configs/generator_selection_evidence_v1.json